<a href="https://colab.research.google.com/github/Maurofome/Portafolio_Actividades_MineriaDatos_-MauricioAliaga/blob/main/Lab2_12_MLP_MLflow/Lab2.12_MLP_MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Paso 1 · Entorno y semilla

In [1]:
!pip install -q mlflow pyngrok

In [2]:
import os, random, subprocess, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, mlflow, kagglehub

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

SEED = 2  # número de lista
random.seed(SEED)
np.random.seed(SEED)

print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("mlflow      ", mlflow.__version__)

pandas       2.2.3
scikit-learn 1.6.1
mlflow       3.18.0


Paso 2 · Carga y preparación de datos

In [3]:
ruta = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
print(os.listdir(ruta))
df = pd.read_csv(os.path.join(ruta, "data.csv"))
print(df.shape)
df.info()

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
['data.csv']
(569, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569

In [4]:
print(df.isna().sum()[df.isna().sum() > 0])   # columna vacía: Unnamed: 32
df = df.drop(columns=["Unnamed: 32", "id"])
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
print(df.shape)
df["diagnosis"].value_counts()

Unnamed: 32    569
dtype: int64
(569, 31)


,count
diagnosis,
0,357
1,212


Hay 357 tumores benignos y 212 malignos. El dataset está algo desbalanceado hacia los benignos, así que la exactitud por sí sola puede ocultar errores en la clase maligna. Por eso también se mide el recall de esa clase.

In [5]:
cols = ["radius_mean", "area_mean", "smoothness_mean"]
escala = df[cols].agg(["min", "max"]).T
escala["rango"] = escala["max"] - escala["min"]
print(escala)
print("\nrango area_mean / rango smoothness_mean =",
      round(escala.loc["area_mean", "rango"] / escala.loc["smoothness_mean", "rango"]))
print("rango area_mean / rango radius_mean    =",
      round(escala.loc["area_mean", "rango"] / escala.loc["radius_mean", "rango"]))

                       min        max       rango
radius_mean        6.98100    28.1100    21.12900
area_mean        143.50000  2501.0000  2357.50000
smoothness_mean    0.05263     0.1634     0.11077

rango area_mean / rango smoothness_mean = 21283
rango area_mean / rango radius_mean    = 112


area_mean va de 143,5 a 2501, radius_mean de 6,98 a 28,11 y smoothness_mean de 0,053 a 0,163. El rango de area_mean es unas 21.000 veces mayor que el de smoothness_mean y unas 112 veces mayor que el de radius_mean. Sin escalar, area_mean pesaría mucho más que smoothness_mean en la suma ponderada de cada neurona.

Paso 3 · Partición estratificada

In [6]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print("Proporción train:\n", y_train.value_counts(normalize=True).round(3))
print("Proporción test:\n",  y_test.value_counts(normalize=True).round(3))

Proporción train:
 diagnosis
0    0.626
1    0.374
Name: proportion, dtype: float64
Proporción test:
 diagnosis
0    0.632
1    0.368
Name: proportion, dtype: float64


Paso 4 · MLflow + ngrok

In [7]:
import requests
from google.colab import userdata
from pyngrok import ngrok

env = dict(os.environ,
           MLFLOW_SERVER_ALLOWED_HOSTS="*",
           MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*")
subprocess.Popen(["mlflow", "server",
                  "--backend-store-uri", "sqlite:///mlflow.db",
                  "--host", "127.0.0.1", "--port", "5000"], env=env)

for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=1).ok:
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill()
tunel = ngrok.connect(5000)
print("MLflow UI:", tunel.public_url)

ReadTimeout: HTTPConnectionPool(host='127.0.0.1', port=5000): Read timed out. (read timeout=1)

Nota: la primera celda falló con ReadTimeout porque el servidor tardó más de 1 s en responder y el bucle solo capturaba ConnectionError. El servidor sí quedó iniciado.

In [8]:
import requests
from google.colab import userdata
from pyngrok import ngrok

# Esperar a que el servidor (ya iniciado) responda
for _ in range(90):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=3).ok:
            print("Servidor MLflow listo")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill()
tunel = ngrok.connect(5000)
print("MLflow UI:", tunel.public_url)

Servidor MLflow listo
MLflow UI: https://scouting-upstream-wad.ngrok-free.dev


In [9]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("IEI097-L2.12-Aliaga")

2026/11/06 22:36:03 INFO mlflow.tracking.fluent: Experiment with name 'IEI097-L2.12-Aliaga' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1794004563540, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794004563540, lifecycle_stage='active', name='IEI097-L2.12-Aliaga', tags={}, trace_location=None, workspace='default'>

Paso 5 · Experimentos
Espero que el escalado mejore el resultado. Las variables tienen rangos muy distintos. Sin escalar, las variables grandes dominan la suma ponderada de cada neurona, pueden saturar las activaciones y hacen que el descenso del gradiente avance de forma desbalanceada. Al estandarizar, todas aportan en igualdad de condiciones y la red debería aprender mejor y de forma más estable.

In [10]:
def correr_experimento(nombre, escalado, capas, activacion, seed_modelo,
                       fase="experimentos", alpha=0.0001, max_iter=500):
    mlp = MLPClassifier(hidden_layer_sizes=capas, activation=activacion,
                        alpha=alpha, max_iter=max_iter, random_state=seed_modelo)
    pasos = [("scaler", StandardScaler())] if escalado else []
    modelo = Pipeline(pasos + [("mlp", mlp)])   # el scaler se ajusta solo con train

    with mlflow.start_run(run_name=nombre):
        mlflow.set_tag("fase", fase)
        modelo.fit(X_train, y_train)
        pred_tr, pred_te = modelo.predict(X_train), modelo.predict(X_test)

        mlflow.log_params({
            "capas": str(capas), "activacion": activacion, "alpha": alpha,
            "max_iter": max_iter, "random_state": seed_modelo,
            "escalado": escalado, "seed_particion": SEED})

        metricas = {
            "accuracy_train": accuracy_score(y_train, pred_tr),
            "accuracy_test":  accuracy_score(y_test, pred_te),
            "recall_maligno": recall_score(y_test, pred_te, pos_label=1),
            "f1":             f1_score(y_test, pred_te, pos_label=1),
            "n_iter":         mlp.n_iter_}
        mlflow.log_metrics(metricas)

        for epoca, loss in enumerate(mlp.loss_curve_):
            mlflow.log_metric("train_loss", loss, step=epoca)

        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_test, pred_te, display_labels=["Benigno", "Maligno"], ax=ax, colorbar=False)
        ax.set_title(nombre)
        mlflow.log_figure(fig, "matriz_confusion.png")
        plt.close(fig)

    print(nombre, {k: round(v, 4) for k, v in metricas.items()})
    return metricas

In [11]:
res_A = correr_experimento("A_sinesc_30_relu", escalado=False, capas=(30,), activacion="relu", seed_modelo=SEED)

🏃 View run A_sinesc_30_relu at: http://127.0.0.1:5000/#/experiments/1/runs/4e00a6ae0efb4afdbadde630527ef7ac
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
A_sinesc_30_relu {'accuracy_train': 0.9451, 'accuracy_test': 0.9474, 'recall_maligno': 0.9524, 'f1': 0.9302, 'n_iter': 187}


Sin escalar, el MLP obtiene accuracy_test = 0,947 y recall_maligno = 0,952 (2 de 42 malignos no detectados). Incluso en entrenamiento queda en 0,945, lo que muestra que la red no logra ajustarse bien a los datos.

In [12]:
res_B = correr_experimento("B_esc_30_relu", escalado=True, capas=(30,), activacion="relu", seed_modelo=SEED)

🏃 View run B_esc_30_relu at: http://127.0.0.1:5000/#/experiments/1/runs/228224851f5b4456807d539342fd63fb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
B_esc_30_relu {'accuracy_train': 0.9934, 'accuracy_test': 0.9737, 'recall_maligno': 0.9762, 'f1': 0.9647, 'n_iter': 310}


accuracy_test sube de 0,947 a 0,974 (de 6 a 3 errores) y recall_maligno de 0,952 a 0,976 (solo 1 falso negativo). La exactitud en train sube de 0,945 a 0,993. Se confirma la hipótesis: con las variables en la misma escala, la red aprende mejor. Hizo más iteraciones (310 contra 187), pero llegó a una solución mucho mejor.

In [13]:
res_C = correr_experimento("C_esc_64-32_relu", escalado=True, capas=(64, 32), activacion="relu", seed_modelo=SEED)

🏃 View run C_esc_64-32_relu at: http://127.0.0.1:5000/#/experiments/1/runs/28580c8f863e4705818e342bd7b51bdf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_esc_64-32_relu {'accuracy_train': 1.0, 'accuracy_test': 0.9737, 'recall_maligno': 0.9524, 'f1': 0.9639, 'n_iter': 176}


accuracy_test se mantiene en 0,974 y recall_maligno baja de 0,976 a 0,952 (un falso negativo más). La exactitud en train llega a 1,0, es decir, la red más grande memoriza el entrenamiento pero no generaliza mejor que B. La diferencia con B es de un solo paciente, así que no se puede decir que una arquitectura sea mejor que la otra.

In [14]:
res_D = correr_experimento("D_esc_64-32_tanh", escalado=True, capas=(64, 32), activacion="tanh", seed_modelo=SEED)

🏃 View run D_esc_64-32_tanh at: http://127.0.0.1:5000/#/experiments/1/runs/640731cea45948cc80b3fb63f748af06
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
D_esc_64-32_tanh {'accuracy_train': 1.0, 'accuracy_test': 0.9825, 'recall_maligno': 0.9762, 'f1': 0.9762, 'n_iter': 271}


con tanh se obtiene el mejor resultado: accuracy_test = 0,982 (2 errores) y recall_maligno = 0,976 (1 falso negativo). Aun así, la mejora frente a C es de un solo tumor, y el run usa una única semilla, así que no basta para afirmar que tanh sea mejor.

In [15]:
res_E = correr_experimento("E_esc_64-32_relu_alpha0.01", escalado=True, capas=(64, 32),
                           activacion="relu", seed_modelo=SEED, alpha=0.01)

🏃 View run E_esc_64-32_relu_alpha0.01 at: http://127.0.0.1:5000/#/experiments/1/runs/afdca1ed0ff14c48b3773c3082cb12f6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
E_esc_64-32_relu_alpha0.01 {'accuracy_train': 1.0, 'accuracy_test': 0.9737, 'recall_maligno': 0.9524, 'f1': 0.9639, 'n_iter': 181}


Las métricas quedan idénticas a C (accuracy_test 0,974, recall 0,952). Solo cambian levemente las iteraciones (181 contra 176). Con este nivel de regularización L2 no hay un efecto visible.

Paso 6 · Semillas y replicación

In [16]:
res_semillas = []
for s in range(SEED, SEED + 5):
    m = correr_experimento(f"C_semilla_{s}", escalado=True, capas=(64, 32),
                           activacion="relu", seed_modelo=s, fase="semillas")
    res_semillas.append({"semilla": s, **m})
tabla_semillas = pd.DataFrame(res_semillas)
tabla_semillas

🏃 View run C_semilla_2 at: http://127.0.0.1:5000/#/experiments/1/runs/cda91b08b7db4f32ac281d1a55035d42
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_semilla_2 {'accuracy_train': 1.0, 'accuracy_test': 0.9737, 'recall_maligno': 0.9524, 'f1': 0.9639, 'n_iter': 176}
🏃 View run C_semilla_3 at: http://127.0.0.1:5000/#/experiments/1/runs/792bc1614fee42239ad78bc6e36cb341
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_semilla_3 {'accuracy_train': 1.0, 'accuracy_test': 0.9737, 'recall_maligno': 0.9524, 'f1': 0.9639, 'n_iter': 184}
🏃 View run C_semilla_4 at: http://127.0.0.1:5000/#/experiments/1/runs/f794590c3d1f4a96aaba69c0a3722696
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_semilla_4 {'accuracy_train': 1.0, 'accuracy_test': 0.9737, 'recall_maligno': 0.9524, 'f1': 0.9639, 'n_iter': 196}
🏃 View run C_semilla_5 at: http://127.0.0.1:5000/#/experiments/1/runs/db0f049eb2434deab3cc51eb307e3b4b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,semilla,accuracy_train,accuracy_test,recall_maligno,f1,n_iter
0,2,1.0,0.973684,0.952381,0.963855,176
1,3,1.0,0.973684,0.952381,0.963855,184
2,4,1.0,0.973684,0.952381,0.963855,196
3,5,1.0,0.973684,0.952381,0.963855,205
4,6,1.0,0.964912,0.952381,0.952381,195


In [17]:
resumen = tabla_semillas[["accuracy_test", "recall_maligno"]].agg(["min", "max"]).T
resumen["rango"] = resumen["max"] - resumen["min"]
resumen

,min,max,rango
accuracy_test,0.964912,0.973684,0.008772
recall_maligno,0.952381,0.952381,0.000000


In [18]:
res_replica = correr_experimento(f"C_replica_semilla_{SEED}", escalado=True, capas=(64, 32),
                                 activacion="relu", seed_modelo=SEED, fase="replica")
comparacion = pd.DataFrame({"run_C": res_C, "replica": res_replica})
comparacion["diferencia"] = comparacion["run_C"] - comparacion["replica"]
comparacion

🏃 View run C_replica_semilla_2 at: http://127.0.0.1:5000/#/experiments/1/runs/7861441df7db4553aea00c8c40a0ad9d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_replica_semilla_2 {'accuracy_train': 1.0, 'accuracy_test': 0.9737, 'recall_maligno': 0.9524, 'f1': 0.9639, 'n_iter': 176}


,run_C,replica,diferencia
accuracy_train,1.000000,1.000000,0.0
accuracy_test,0.973684,0.973684,0.0
recall_maligno,0.952381,0.952381,0.0
f1,0.963855,0.963855,0.0
n_iter,176.000000,176.000000,0.0


Variabilidad (5 semillas): accuracy_test va de 0,965 a 0,974 (rango 0,009, o sea, 1 tumor de diferencia) y recall_maligno se mantiene en 0,952 en las 5 semillas (rango 0). Cambiar solo los pesos iniciales mueve poco el resultado, pero lo mueve.

Replicación: C_replica_semilla_2 da exactamente las mismas métricas que el run C (diferencia 0,0 en todas, incluso n_iter = 176). Con la semilla fija, el experimento es repetible.

Paso 7 · Análisis en MLflow y exportación

In [19]:
runs = mlflow.search_runs()
columnas = ["tags.mlflow.runName", "tags.fase",
            "params.escalado", "params.capas", "params.activacion", "params.alpha", "params.random_state",
            "metrics.accuracy_train", "metrics.accuracy_test", "metrics.recall_maligno",
            "metrics.f1", "metrics.n_iter"]
runs = runs[columnas].sort_values("metrics.recall_maligno", ascending=False)
runs.to_csv("runs.csv", index=False)
print(len(runs), "runs")
runs

11 runs


,tags.mlflow.runName,tags.fase,params.escalado,params.capas,params.activacion,params.alpha,params.random_state,metrics.accuracy_train,metrics.accuracy_test,metrics.recall_maligno,metrics.f1,metrics.n_iter
9,B_esc_30_relu,experimentos,True,"(30,)",relu,0.0001,2,0.993407,0.973684,0.976190,0.964706,310.0
7,D_esc_64-32_tanh,experimentos,True,"(64, 32)",tanh,0.0001,2,1.000000,0.982456,0.976190,0.976190,271.0
0,C_replica_semilla_2,replica,True,"(64, 32)",relu,0.0001,2,1.000000,0.973684,0.952381,0.963855,176.0
2,C_semilla_5,semillas,True,"(64, 32)",relu,0.0001,5,1.000000,0.973684,0.952381,0.963855,205.0
1,C_semilla_6,semillas,True,"(64, 32)",relu,0.0001,6,1.000000,0.964912,0.952381,0.952381,195.0
3,C_semilla_4,semillas,True,"(64, 32)",relu,0.0001,4,1.000000,0.973684,0.952381,0.963855,196.0
4,C_semilla_3,semillas,True,"(64, 32)",relu,0.0001,3,1.000000,0.973684,0.952381,0.963855,184.0
6,E_esc_64-32_relu_alpha0.01,experimentos,True,"(64, 32)",relu,0.01,2,1.000000,0.973684,0.952381,0.963855,181.0
5,C_semilla_2,semillas,True,"(64, 32)",relu,0.0001,2,1.000000,0.973684,0.952381,0.963855,176.0
8,C_esc_64-32_relu,experimentos,True,"(64, 32)",relu,0.0001,2,1.000000,0.973684,0.952381,0.963855,176.0


In [20]:
from google.colab import files
files.download("runs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

¿Qué cambió al escalar las variables?
Al escalar (de A_sinesc_30_relu a B_esc_30_relu), accuracy_test subió de 0,947 a 0,974 y recall_maligno de 0,952 a 0,976. Sin escalar, variables grandes como area_mean dominan sobre las pequeñas como smoothness_mean y la red aprende peor. Con StandardScaler todas quedan en la misma escala.

¿Más capas y neuronas mejoraron el resultado?
No. El run C (64, 32) tuvo la misma exactitud que B (0,974) y menor recall (0,952). La diferencia es de un solo paciente, igual a la variación entre semillas, así que no se nota una mejora real.

¿Qué tan grande fue la variación con las 5 semillas?
Pequeña: la exactitud varió entre 0,965 y 0,974 y el recall no cambió (0,952). Es menor que el efecto del escalado, pero igual a las diferencias entre B, C y D. Por eso no puedo decir que una arquitectura sea “mejor” con un solo run.

¿Qué error es más costoso?
El falso negativo, porque deja sin tratamiento a un paciente con cáncer. Por eso priorizo el recall de la clase maligna.

¿Qué no tendría solo con print()?
Con print() los resultados se pierden y no se pueden comparar. MLflow guarda parámetros, métricas, la curva de pérdida y la matriz de confusión de cada run, y permite compararlos. Para replicar el experimento registraría también las versiones de las librerías y el modelo entrenado.

In [23]:
ngrok.kill()

In [22]:
tunel = ngrok.connect(5000)
print("MLflow UI:", tunel.public_url)

MLflow UI: https://scouting-upstream-wad.ngrok-free.dev
